In [ ]:
# bowaka_v2_lab notebook bootstrap cell — DO NOT EDIT BY HAND.
# Adds the lab's src/ (and its bowaka_common dependency) to sys.path and pins
# the working directory to the repo root, so `import bowaka_v2_lab` and
# repo-root-relative CONFIG_PATH parameters resolve identically under jupyter,
# papermill, and the QuantsLab scheduler.
import os
import sys
from pathlib import Path

_lab_root = None
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "bowaka_v2_lab" / "__init__.py").is_file():
        _lab_root = _candidate
        break
if _lab_root is None:
    raise RuntimeError(
        f"bowaka_v2_lab bootstrap: src/bowaka_v2_lab/ not found at or above {Path.cwd()}"
    )

# Pin CWD to the repo root (the directory holding research_notebooks/ and the
# Makefile) so repo-root-relative CONFIG_PATH values resolve regardless of how
# the notebook was launched (jupyter CWD = notebook dir, scheduler = repo root).
_repo_root = _lab_root
for _candidate in [_lab_root, *_lab_root.parents]:
    if (_candidate / "research_notebooks").is_dir() and (_candidate / "Makefile").is_file():
        _repo_root = _candidate
        break
os.chdir(_repo_root)

# Make the lab and its bowaka_common dependency importable from the working
# tree, even when the packages are not pip-installed. v1 bowaka_lab is
# deliberately excluded — v2 must not import v1.
for _src in (_lab_root / "src",
             _repo_root / "research_notebooks" / "bowaka_common" / "src"):
    if _src.is_dir() and str(_src) not in sys.path:
        sys.path.insert(0, str(_src))

import bowaka_v2_lab  # noqa: F401
print(f"bowaka_v2_lab {bowaka_v2_lab.__version__} (cwd={_repo_root})")


In [ ]:
# Papermill parameter cell.
CONFIG_PATH = 'research_notebooks/bowaka_v2_lab/configs/bowaka_v2_walkforward_optuna.yml'
N_TRIALS = 5  # smoke; raise to 200+ for real runs


# 10 — Optuna Walk-Forward

Runs a small TPE study against the SIP config.

In [ ]:
import optuna
from bowaka_v2_lab.optuna.dispatcher import OptunaStudy
from bowaka_v2_lab.optuna.search_space import suggest_params
from bowaka_v2_lab.optuna.objective import compute_objective, FoldResult
study = OptunaStudy(feed='sip', cost_stress='conservative',
                      dataset_hash='cafebabecafebabe', config_hash='deadbeefdeadbeef',
                      n_trials=N_TRIALS)
study.create()

def objective(trial):
    params = suggest_params(trial)
    # Synthetic single-fold: penalise extreme params.
    fold = FoldResult(fold_id='f0',
        net_return=0.01 - abs(params['signals.gap_pct_max'] - 0.10) * 0.1,
        max_drawdown=0.02, turnover=0.0, concentration=0.0,
        n_trades=8, ambiguous_bar_count=0, missing_quote_count=0)
    return compute_objective([fold]).objective

study.optimize(objective)
print('best value:', study.study.best_value)
print('best params:', study.study.best_params)
